# Week 13A — Vision–Language Models and Auto-Labelling

**MCTA 4364 Machine Vision** · Session 1 of Week 13 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W13_vision_language/W13A_vlm_auto_labelling.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W13_vision_language/W13A_vlm_auto_labelling.ipynb)

Every model so far needed a fixed list of classes and a labelled dataset. **Vision–language models (VLMs)** learn from hundreds of millions of
image–caption pairs from the web, so you can *describe* a class in words instead of labelling it. Today you will measure what that buys you — and
where it breaks — on natural images and on our conveyor parts, then use an open-vocabulary detector to **auto-label** a detection dataset from
four clicks, and check whether a detector trained on those labels is as good as one trained on human labels (the data engine of Week 9A).

### Learning outcomes
By the end of this session you will be able to:
1. Explain **contrastive image–text pre-training** (CLIP) and compute its similarity matrix and loss.
2. Use CLIP for **zero-shot classification** and **text-to-image retrieval**, and engineer prompts (templates, ensembles).
3. Measure the **domain gap** of zero-shot models on industrial images and close it with a **few-shot linear probe**.
4. Run an **open-vocabulary detector** (YOLOE) with **text** and **visual** prompts.
5. Build an **auto-labelling pipeline** with confidence-based **human review**, measure label quality, and train on the result.
6. Describe **generative VLMs** (captioning, VQA, structured output), their failure modes (**hallucination**) and safeguards.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–12 | 1 | Contrastive pre-training: a measured similarity matrix |
| 12–30 | 2–3 | Zero-shot classification, prompt engineering, the domain gap (interactive) |
| 30–40 | 4–5 | Few-shot linear probe; text-to-image search (interactive) |
| 40–62 | 6–8 | Open-vocabulary detection; auto-labelling with human review; training on auto-labels (starts while we discuss) |
| 62–72 | 9 | Generative VLMs, hallucination and safeguards |
| 72–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, challenge |

**Hardware:** everything runs on a laptop CPU (the two small detector trainings in Section 8 take ~10 min on a CPU, ~1–2 min on a 4 GB GPU).
Downloads: MobileCLIP (~600 MB) and YOLOE-11S (~30 MB) from GitHub, once; optional SmolVLM (~0.5 GB) from Hugging Face in Section 9.

> **Licences.** MobileCLIP weights: Apple (research and education). YOLOE / Ultralytics: **AGPL-3.0** (see Week 8B for the implications and
> permissive alternatives). SmolVLM: Apache-2.0. OpenAI CLIP: MIT.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "open_clip": "open_clip_torch", "ultralytics": "ultralytics"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import shutil
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data
import torch

os.environ.setdefault("YOLO_VERBOSE", "False")                  # quieter Ultralytics output (no progress bars)
from ultralytics import YOLO, YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor
from ultralytics.utils import ASSETS

import partsim
from cvhelpers import SMOKE, check, data_dir, download, interact, load_sample, pipeline_diagram, quiz
from dlhelpers import confusion_matrix, hw_profile, plot_confusion, set_seed
from vlmhelpers import load_clip, yoloe_set_text_classes

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = 0 if HW["cuda"] else "cpu"
set_seed(0)
WEIGHTS = data_dir("weights")
t0 = time.perf_counter()
clip_model = load_clip("mobileclip")
print(f"{clip_model.name}: {clip_model.dim}-D embeddings, logit scale {clip_model.logit_scale:.0f}, loaded in {time.perf_counter() - t0:.0f} s")


def rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img[..., ::-1]


def bgr(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img[..., ::-1].copy()

## 1. Contrastive image–text pre-training (CLIP)

**CLIP** (Radford et al., 2021) has two encoders: an **image encoder** $f$ (a ViT or CNN) and a **text encoder** $g$ (a transformer). Both map
into the same $D$-dimensional space and their outputs are normalised to unit length, so the **cosine similarity** is a dot product:

$$s_{ij} = \mathbf{u}_i^\top \mathbf{v}_j, \qquad \mathbf{u}_i = \frac{f(x_i)}{\lVert f(x_i)\rVert}, \quad \mathbf{v}_j = \frac{g(t_j)}{\lVert g(t_j)\rVert}.$$

Training uses batches of $N$ matching image–caption pairs. The $N\times N$ similarity matrix should be large on the **diagonal** (the true pairs)
and small elsewhere. The **InfoNCE** loss is a cross-entropy in both directions, with a learned temperature $\tau$ (logit scale $1/\tau$):

$$\mathcal{L} = -\frac{1}{2N}\sum_{i=1}^{N}\left[\log\frac{e^{s_{ii}/\tau}}{\sum_{j} e^{s_{ij}/\tau}} + \log\frac{e^{s_{ii}/\tau}}{\sum_{j} e^{s_{ji}/\tau}}\right].$$

With 400 million pairs (CLIP) to billions (SigLIP 2, MobileCLIP's DataCompDR), the model learns visual concepts **named in natural language** —
without anyone defining a class list. We use **MobileCLIP-B** (Vasu et al., 2024): the same quality as a large CLIP at a fraction of the cost, and
the text encoder that the YOLOE detector of Section 6 uses.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["image", "image encoder\n(ViT-B/16)", "u (512-D, unit)", "cosine\nsimilarity u . v", "v (512-D, unit)", "text encoder\n(transformer)", "caption"],
                 colors=["#f1f3f4", "#e8f0fe", "#e6f4ea", "#fef7e0", "#e6f4ea", "#fce8e6", "#f1f3f4"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - Two encoders, one shared embedding space", fontweight="bold")
plt.show()

In [ ]:
pairs = [
    (bgr(skimage.data.astronaut()), "an astronaut in a white space suit"),
    (bgr(skimage.data.chelsea()), "a ginger cat"),
    (bgr(skimage.data.coffee()), "a cup of coffee on a saucer"),
    (bgr(skimage.data.coins()), "old coins on a table"),
    (bgr(skimage.data.rocket()), "a rocket on a launch pad"),
    (partsim.make_scene(3, n_objects=5)[0], "metal parts on a conveyor belt"),
]
U = clip_model.encode_images([p[0] for p in pairs])
V = clip_model.encode_text([p[1] for p in pairs])
Ssim = U @ V.T
logits = clip_model.logit_scale * Ssim


def log_softmax(z, axis):
    z = z - z.max(axis=axis, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=axis, keepdims=True))


loss = -(np.trace(log_softmax(logits, 1)) + np.trace(log_softmax(logits, 0))) / (2 * len(pairs))
fig = plt.figure(figsize=(22, 7))
for k, (im, cap_) in enumerate(pairs):
    ax = fig.add_axes([0.02, 0.84 - k * 0.155, 0.1, 0.14]); ax.imshow(rgb(im)); ax.axis("off")
ax = fig.add_axes([0.16, 0.08, 0.36, 0.9])
im_ = ax.imshow(Ssim, cmap="viridis")
for i in range(len(pairs)):
    for j in range(len(pairs)):
        ax.text(j, i, f"{Ssim[i, j]:.2f}", ha="center", va="center", color="w" if Ssim[i, j] < Ssim.max() * 0.7 else "k")
ax.set_xticks(range(len(pairs))); ax.set_xticklabels([p[1] for p in pairs], rotation=25, ha="right"); ax.set_yticks([])
ax.set_title("cosine similarity: images (rows) x captions (columns)")
plt.colorbar(im_, ax=ax, fraction=0.04)
ax = fig.add_axes([0.62, 0.3, 0.36, 0.6])
P_ = np.exp(log_softmax(logits, 1))
ax.imshow(P_, cmap="Greens", vmin=0, vmax=1)
for i in range(len(pairs)):
    for j in range(len(pairs)):
        ax.text(j, i, f"{P_[i, j]:.2f}", ha="center", va="center")
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"softmax over captions (scale {clip_model.logit_scale:.0f}): InfoNCE loss = {loss:.3f}")
fig.suptitle("Figure 1.2 - The CLIP objective, measured: the true pairs win by a small cosine margin that the scale turns into near-certainty",
             fontweight="bold", y=1.07)
plt.show()
print(f"diagonal mean {np.diag(Ssim).mean():.3f} vs off-diagonal mean {Ssim[~np.eye(len(pairs), dtype=bool)].mean():.3f}")

**Read the numbers.** Absolute cosine similarities are small (≈ 0.2–0.3 for a match, ≈ 0 for a mismatch): the two modalities occupy different
regions of the space (the *modality gap*). Only the **ranking** matters, and the logit scale (≈ 60–100) turns a margin of 0.2 into a confident
softmax. Do not read a single cosine value as a probability.

## 2. Zero-shot classification

To classify an image into classes $c = 1..C$, write a **prompt** for each class ("a photo of a {class}"), encode the prompts once, and pick the
class whose text embedding is closest:

$$p(c \mid x) = \frac{\exp(\lambda\, \mathbf{u}^\top \mathbf{v}_c)}{\sum_{k} \exp(\lambda\, \mathbf{u}^\top \mathbf{v}_k)}, \qquad \lambda = \text{logit scale}.$$

The text embeddings **are** the weights of a linear classifier — written in English instead of learned from labels.

In [ ]:
def zero_shot_probs_ref(img_emb, txt_emb, scale):
    z = scale * np.atleast_2d(img_emb) @ txt_emb.T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


GALLERY = {
    "astronaut": bgr(skimage.data.astronaut()), "cat": bgr(skimage.data.chelsea()), "coffee": bgr(skimage.data.coffee()),
    "coins": bgr(skimage.data.coins()), "rocket": bgr(skimage.data.rocket()), "camera man": bgr(skimage.data.camera()),
    "clock": bgr(skimage.data.clock()), "galaxies": bgr(skimage.data.hubble_deep_field()), "tissue": bgr(skimage.data.immunohistochemistry()),
    "moon": bgr(skimage.data.moon()), "page": bgr(skimage.data.page()), "fruits": load_sample("fruits.jpg"), "baboon": load_sample("baboon.jpg"),
    "football": load_sample("messi5.jpg"), "house": load_sample("home.jpg"), "building": load_sample("building.jpg"),
    "painting": load_sample("starry_night.jpg"), "butterfly": load_sample("butterfly.jpg"), "sweets": load_sample("smarties.png"),
    "orange": load_sample("orange.jpg"), "sudoku": load_sample("sudoku.png"), "fish": load_sample("HappyFish.jpg"),
    "boxes": load_sample("box_in_scene.png"), "plant": load_sample("aloeL.jpg"), "circuit board": load_sample("board.jpg"),
    "bus": cv2.imread(str(ASSETS / "bus.jpg")), "people": cv2.imread(str(ASSETS / "zidane.jpg")),
    "conveyor 1": partsim.make_scene(11, n_objects=5)[0], "conveyor 2": partsim.make_scene(12, n_objects=3, exposure=0.6)[0],
    "part ok": partsim.make_part(1, "ok")[0], "part scratch": partsim.make_part(2, "scratch")[0], "part chip": partsim.make_part(3, "chip")[0],
}
t0 = time.perf_counter()
G_EMB = clip_model.encode_images(list(GALLERY.values()))
print(f"encoded {len(GALLERY)} gallery images in {time.perf_counter() - t0:.1f} s ({clip_model.dim}-D each)")

LABELS = ["person", "animal", "food or drink", "vehicle", "building", "machine part", "text document", "space", "artwork", "microscope image"]
T_EMB = clip_model.encode_text([f"a photo of a {c}." for c in LABELS])
show_names = ["astronaut", "cat", "coffee", "bus", "house", "painting", "page", "tissue", "conveyor 1", "part scratch"]
fig, axes = plt.subplots(2, 10, figsize=(26, 6.2), gridspec_kw=dict(height_ratios=[1.2, 1]))
for k, name in enumerate(show_names):
    p = zero_shot_probs_ref(G_EMB[list(GALLERY).index(name)], T_EMB, clip_model.logit_scale)[0]
    top = np.argsort(-p)[:3]
    axes[0, k].imshow(rgb(GALLERY[name])); axes[0, k].axis("off"); axes[0, k].set_title(name)
    axes[1, k].barh(range(3)[::-1], p[top], color=["#1a73e8", "#9aa0a6", "#dadce0"]); axes[1, k].set_xlim(0, 1)
    axes[1, k].set_yticks(range(3)[::-1]); axes[1, k].set_yticklabels([LABELS[t] for t in top], fontsize=8)
fig.suptitle("Figure 2.1 - Zero-shot classification with 10 classes written as text (top-3 probabilities)", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: write your own classes
Type comma-separated class names and a template (`{}` is replaced by each name). Try synonyms, negatives ("a blurry photo"), and absurd classes.

In [ ]:
def zero_shot_demo(image="coffee", classes="coffee, tea, soup, a cat, a car", template="a photo of {}."):
    names = [c.strip() for c in classes.split(",") if c.strip()]
    if not names:
        return
    p = zero_shot_probs_ref(G_EMB[list(GALLERY).index(image)], clip_model.encode_text([template.format(c) for c in names]), clip_model.logit_scale)[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw=dict(width_ratios=[1, 1.6]))
    axes[0].imshow(rgb(GALLERY[image])); axes[0].axis("off")
    order = np.argsort(p)
    axes[1].barh([names[i] for i in order], p[order], color="#1a73e8"); axes[1].set_xlim(0, 1); axes[1].set_xlabel("probability")
    axes[1].set_title(f"template: '{template}'")
    plt.tight_layout(); plt.show()


interact(zero_shot_demo, image=widgets.Dropdown(options=list(GALLERY), value="coffee"),
         classes=widgets.Text(value="coffee, tea, soup, a cat, a car", layout=widgets.Layout(width="70%")),
         template=widgets.Text(value="a photo of {}."))

## 3. Prompt engineering and the domain gap

Zero-shot accuracy depends on **the words you choose** — the model saw captions written by people on the web, not your class names.
Two standard tricks:
- **templates** that match the image style ("a close-up photo of a metal {}", "a grey-scale photo of a {} on a conveyor belt");
- **prompt ensembling**: average the (unit) text embeddings of many templates per class, then renormalise — usually +1–5 %.

We test on single-part crops from the conveyor (Week 9B's four classes) and then on the **defect** images of Week 7A.

In [ ]:
N_CROP = 12 if SMOKE else 60
crops, crop_y = partsim.make_crops(N_CROP, seed=0, size=96)
CROP_EMB = clip_model.encode_images(list(crops))
PART_NAMES = {"short names": ["hex nut", "washer", "bolt", "flange"],
              "descriptive": ["hexagonal nut", "flat round washer", "bolt with a threaded shank", "pipe flange with bolt holes"]}
TEMPLATES = ["a photo of a {}.", "a close-up photo of a metal {}.", "a grey-scale photo of a {} on a conveyor belt.", "a {}.",
             "an industrial photo of a {}.", "a photo of a steel {}, top view."]


def prompt_ensemble_ref(encode_text, classnames, templates):
    out = []
    for c in classnames:
        e = encode_text([t.format(c) for t in templates])
        e = e / np.linalg.norm(e, axis=1, keepdims=True)
        m = e.mean(0)
        out.append(m / np.linalg.norm(m))
    return np.stack(out)


results = {}
for nk, names in PART_NAMES.items():
    for t in TEMPLATES:
        results[(nk, t)] = np.mean((CROP_EMB @ clip_model.encode_text([t.format(n) for n in names]).T).argmax(1) == crop_y)
    results[(nk, "ENSEMBLE of all templates")] = np.mean((CROP_EMB @ prompt_ensemble_ref(clip_model.encode_text, names, TEMPLATES).T).argmax(1) == crop_y)
fig, axes = plt.subplots(1, 2, figsize=(22, 4.8), gridspec_kw=dict(width_ratios=[1, 2.2]))
axes[0].imshow(np.vstack([np.hstack([crops[crop_y == c][j] for j in range(3)]) for c in range(4)]), cmap="gray"); axes[0].axis("off")
axes[0].set_title("test crops (rows: " + ", ".join(PART_NAMES["short names"]) + ")")
labels_ = list(dict.fromkeys(t for _, t in results))
xs = np.arange(len(labels_))
for off, (nk, col) in zip([-0.2, 0.2], [("short names", "#1a73e8"), ("descriptive", "#f29900")]):
    axes[1].bar(xs + off, [results[(nk, t)] for t in labels_], 0.4, color=col, label=f"class names: {nk}")
axes[1].axhline(0.25, color="k", ls=":", label="chance (4 classes)")
axes[1].set_xticks(xs); axes[1].set_xticklabels([t.replace(" {}", "\n{}") for t in labels_], fontsize=8); axes[1].set_ylim(0, 1)
axes[1].set_ylabel("zero-shot accuracy"); axes[1].legend(loc="lower right"); axes[1].grid(alpha=0.3, axis="y")
fig.suptitle(f"Figure 3.1 - The same images, different words: zero-shot accuracy on {len(crop_y)} part crops", fontweight="bold")
plt.tight_layout(); plt.show()
best_ = max(results, key=results.get)
print(f"best: {results[best_]:.0%} with {best_}; worst: {min(results.values()):.0%}")

In [ ]:
N_DEF_TR, N_DEF_TE = (8, 8) if SMOKE else (30, 40)


def defect_set(n, seed):
    imgs, ys = [], []
    for ci, d in enumerate(partsim.CLASSES):
        for k in range(n):
            imgs.append(partsim.make_part(seed + 1000 * ci + k, d)[0]); ys.append(ci)
    return imgs, np.array(ys)


t0 = time.perf_counter()
def_tr_img, def_tr_y = defect_set(N_DEF_TR, 0)
def_te_img, def_te_y = defect_set(N_DEF_TE, 50000)
DEF_TR, DEF_TE = clip_model.encode_images(def_tr_img), clip_model.encode_images(def_te_img)
print(f"encoded {len(def_tr_y) + len(def_te_y)} part images in {time.perf_counter() - t0:.0f} s")
DEFECT_PROMPTS = ["a photo of a flawless machined metal part.", "a photo of a metal part with a missing hole.",
                  "a photo of a metal part with a scratch.", "a photo of a metal part with a chipped edge.", "a photo of an undersized metal part."]
zs_pred = (DEF_TE @ clip_model.encode_text(DEFECT_PROMPTS).T).argmax(1)
zs_acc = np.mean(zs_pred == def_te_y)
fig, axes = plt.subplots(1, 6, figsize=(26, 4.8), gridspec_kw=dict(width_ratios=[1, 1, 1, 1, 1, 2.2]))
for ci in range(5):
    axes[ci].imshow(rgb(def_te_img[list(def_te_y).index(ci)])); axes[ci].axis("off"); axes[ci].set_title(partsim.CLASSES[ci])
plot_confusion(confusion_matrix(def_te_y, zs_pred, 5), partsim.CLASSES, title=f"zero-shot: accuracy {zs_acc:.0%} (chance 20%)", ax=axes[5])
fig.suptitle("Figure 3.2 - The domain gap: web captions rarely describe a 0.5 mm chip, so zero-shot defect classification is barely above chance",
             fontweight="bold")
plt.tight_layout(); plt.show()

**Why?** Object *categories* (nut, washer, bolt) appear in millions of web captions; *subtle, domain-specific states* (a chip on the rim, a part 1 mm
too small) almost never do. Zero-shot VLMs are excellent at **what** is in the image and weak at **fine, measured** properties — exactly what
industrial inspection needs. Undersize is invisible to them by construction: without a scale reference, a small part looks like a part.

## 4. Few-shot: a linear probe on CLIP features

The image embeddings still contain much of the information, even when the text side does not. With $k$ labelled examples per class we can
- average them into a **prototype** per class (nearest class mean), or
- train a **linear probe**: multinomial logistic regression on the frozen 512-D embeddings (seconds on a CPU).

This is transfer learning (Week 6B) with a much stronger backbone, and it is how most "foundation model" products adapt to a customer's data.

In [ ]:
def linear_probe(X, y, n_classes, weight_decay=1e-3, steps=200):
    Xt, yt = torch.tensor(X), torch.tensor(y)
    W = torch.zeros(X.shape[1], n_classes, requires_grad=True)
    b = torch.zeros(n_classes, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], max_iter=steps)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(30 * Xt @ W + b, yt) + weight_decay * (W ** 2).sum()
        loss.backward()
        return loss

    opt.step(closure)
    return lambda Z: (30 * torch.tensor(Z) @ W + b).argmax(1).numpy()


SHOTS = [1, 2, 5] if SMOKE else [1, 2, 5, 10, 20, 30]
acc_proto, acc_probe = [], []
for k in SHOTS:
    a1, a2 = [], []
    for s in range(3 if SMOKE else 5):
        r = np.random.default_rng(s)
        idx = np.concatenate([r.choice(np.where(def_tr_y == c)[0], k, replace=False) for c in range(5)])
        proto = np.stack([DEF_TR[idx][def_tr_y[idx] == c].mean(0) for c in range(5)])
        proto /= np.linalg.norm(proto, axis=1, keepdims=True)
        a1.append(np.mean((DEF_TE @ proto.T).argmax(1) == def_te_y))
        a2.append(np.mean(linear_probe(DEF_TR[idx], def_tr_y[idx], 5)(DEF_TE) == def_te_y))
    acc_proto.append(np.mean(a1)); acc_probe.append(np.mean(a2))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(SHOTS, acc_probe, "o-", color="#1a73e8", lw=2.5, label="linear probe (logistic regression on embeddings)")
ax.plot(SHOTS, acc_proto, "s-", color="#f29900", lw=2, label="prototypes (nearest class mean)")
ax.axhline(zs_acc, color="#d93025", ls="--", label=f"zero-shot text prompts ({zs_acc:.0%})")
ax.axhline(0.2, color="k", ls=":", label="chance")
ax.set_xscale("log"); ax.set_xticks(SHOTS); ax.set_xticklabels(SHOTS); ax.set_ylim(0, 1)
ax.set_xlabel("labelled examples per class (k)"); ax.set_ylabel("test accuracy (5 defect classes)"); ax.grid(alpha=0.3); ax.legend(loc="upper left")
ax.set_title("Figure 4.1 - A few labels close most of the domain gap (mean of repeated random draws)", fontweight="bold")
plt.show()
print("k:", SHOTS, "| probe:", np.round(acc_probe, 2), "| prototypes:", np.round(acc_proto, 2))

Compare with **Week 7A**: the aligned golden-template pipeline reached ~99 % on the same defects with no deep learning at all, because it measures
exactly what matters (geometry, the difference to a golden part). General embeddings are a fast *starting point*, not automatically the best
inspection method — keep the classical tools in your kit.

## 5. Text-to-image search

The same embeddings give **semantic search**: encode a query sentence and rank the gallery by cosine similarity. In practice the image embeddings
are stored in a vector index (FAISS, a vector database) and queried in milliseconds — useful to find "all images with a person near the robot"
in a year of logged camera data, to **mine rare cases** for labelling (Week 9A), or to spot near-duplicates (image-to-image similarity).

In [ ]:
def search(query, k=5):
    s = G_EMB @ clip_model.encode_text([query])[0]
    top = np.argsort(-s)[:k]
    return [(list(GALLERY)[i], float(s[i])) for i in top]


QUERIES = ["something to drink", "a vehicle for public transport", "an animal with fur", "a scientific image",
           "washers and nuts on a conveyor", "a board game or puzzle"]
fig, axes = plt.subplots(len(QUERIES), 5, figsize=(17, 3.0 * len(QUERIES)))
for r_, q in enumerate(QUERIES):
    for c_, (name, s) in enumerate(search(q)):
        axes[r_, c_].imshow(rgb(GALLERY[name])); axes[r_, c_].set_xticks([]); axes[r_, c_].set_yticks([])
        axes[r_, c_].set_title(f"{name} ({s:.2f})", fontsize=9)
    axes[r_, 0].set_ylabel(f'"{q}"', rotation=0, ha="right", va="center", fontsize=10)
fig.suptitle(f"Figure 5.1 - Text-to-image search over {len(GALLERY)} images (top 5, cosine similarity)", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: search the gallery

In [ ]:
def search_demo(query="a round object"):
    res = search(query, 6)
    fig, axes = plt.subplots(1, 6, figsize=(20, 3.5))
    for ax, (name, s) in zip(axes, res):
        ax.imshow(rgb(GALLERY[name])); ax.axis("off"); ax.set_title(f"{name} ({s:.2f})")
    plt.suptitle(f'"{query}"'); plt.show()


interact(search_demo, query=widgets.Text(value="a round object", layout=widgets.Layout(width="60%")))

## 6. Open-vocabulary detection

A zero-shot *classifier* says what is in the image; an **open-vocabulary detector** says **where**, for classes given at run time.
**YOLOE** (Wang et al., 2025) is a YOLO11 detector whose classification head compares every candidate box's embedding with **prompt embeddings**:
- **text prompts** — class names encoded by the MobileCLIP text encoder of Sections 1–5;
- **visual prompts** — draw one box around an example in a reference image; the embedding of that region becomes the class;
- **prompt-free** — a built-in vocabulary of ~4,500 everyday categories.

Related models: YOLO-World, Grounding DINO, OWLv2, and SAM 3 (text-prompted segmentation, 2025).

In [ ]:
yoloe = YOLOE(str(download("https://github.com/ultralytics/assets/releases/download/v8.3.0/yoloe-11s-seg.pt", WEIGHTS / "yoloe-11s-seg.pt")))


def draw_dets(img, r, names, color=(0, 200, 0)):
    out = img.copy()
    thickness, fs = max(2, img.shape[1] // 250), max(0.5, img.shape[1] / 900)
    for (x0, y0, x1, y1), c, s in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()):
        cv2.rectangle(out, (x0, y0), (x1, y1), color, thickness)
        cv2.putText(out, f"{names[c]} {s:.2f}", (x0, max(int(20 * fs), y0 - 4)), cv2.FONT_HERSHEY_SIMPLEX, fs, color, thickness)
    return out


examples = [(GALLERY["bus"], ["person", "bus", "traffic light", "backpack"]), (GALLERY["coffee"], ["cup", "spoon", "saucer"]),
            (GALLERY["football"], ["soccer ball", "football player", "goal"]), (GALLERY["conveyor 1"], partsim.SCENE_CLASSES)]
fig, axes = plt.subplots(1, 4, figsize=(26, 5.5))
for ax, (img, names) in zip(axes, examples):
    yoloe_set_text_classes(yoloe, names, clip_model)
    r = yoloe.predict(img, conf=0.2, verbose=False, device=DEVICE)[0]
    ax.imshow(rgb(draw_dets(img, r, names))); ax.axis("off"); ax.set_title("prompts: " + ", ".join(names), fontsize=10)
fig.suptitle("Figure 6.1 - YOLOE with text prompts: good on everyday scenes; on our conveyor, weak scores and the wrong names", fontweight="bold")
plt.tight_layout(); plt.show()

On the conveyor the detector **finds** most parts but calls nearly everything "hex_nut" with low confidence — the text prompts do not describe these
rendered parts well. A **visual prompt** fixes that: show the model one example of each class.

In [ ]:
rng_ = np.random.default_rng(123)
while True:                                                     # a reference scene that contains all four classes
    ref_img, ref_boxes, ref_labels = partsim.make_scene(rng_, 6)
    if set(ref_labels) == set(range(4)):
        break
sel_ = [list(ref_labels).index(c) for c in range(4)]
VISUAL_PROMPT = dict(bboxes=np.asarray(ref_boxes, float)[sel_], cls=np.arange(4))
NAMES = partsim.SCENE_CLASSES


def predict_visual(img, conf=0.05):
    r = yoloe.predict(img, refer_image=ref_img, visual_prompts=VISUAL_PROMPT, predictor=YOLOEVPSegPredictor, conf=conf, verbose=False, device=DEVICE)[0]
    return r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r


def predict_text(img, conf=0.05):
    yoloe_set_text_classes(yoloe, NAMES, clip_model)             # visual-prompt runs replace the classes, so set them again
    r = yoloe.predict(img, conf=conf, verbose=False, device=DEVICE)[0]
    return r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r


test_scene, test_boxes, test_labels = partsim.make_scene(777, 6)
yoloe_set_text_classes(yoloe, NAMES, clip_model)
_, _, _, r_txt = predict_text(test_scene, conf=0.2)
_, _, _, r_vis = predict_visual(test_scene, conf=0.25)
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
axes[0].imshow(rgb(partsim.draw_boxes(ref_img, np.asarray(ref_boxes)[sel_], np.arange(4)))); axes[0].set_title("reference image: ONE box per class (4 clicks)")
axes[1].imshow(rgb(draw_dets(test_scene, r_txt, NAMES, (0, 0, 255)))); axes[1].set_title("new scene, text prompts")
axes[2].imshow(rgb(draw_dets(test_scene, r_vis, NAMES))); axes[2].set_title("new scene, visual prompts")
for ax in axes:
    ax.axis("off")
fig.suptitle("Figure 6.2 - Visual prompting: the example boxes define the classes", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def box_iou(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    x0 = np.maximum(a[:, None, 0], b[None, :, 0]); y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2]); y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def label_errors_ref(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_thr=0.5):
    """Greedy matching in order of score: a prediction is correct if it overlaps an unmatched ground-truth box of the SAME class."""
    iou = box_iou(pred_boxes, gt_boxes)
    used, tp = set(), 0
    for i in np.argsort(-np.asarray(pred_scores)):
        cand = [j for j in range(len(gt_boxes)) if j not in used and gt_labels[j] == pred_labels[i] and iou[i, j] >= iou_thr]
        if cand:
            used.add(max(cand, key=lambda j: iou[i, j])); tp += 1
    return tp, len(pred_boxes) - tp, len(gt_boxes) - tp


N_EVAL = 6 if SMOKE else 30
ev_rng = np.random.default_rng(0)
ev = [partsim.make_scene(ev_rng, 5) for _ in range(N_EVAL)]
curves = {}
for mode, fn in [("text prompts", predict_text), ("visual prompts", predict_visual)]:
    preds_ = [fn(img)[:3] for img, _, _ in ev]
    rows = []
    for thr in np.linspace(0.05, 0.8, 16):
        tp = fp = fn_ = 0
        for (b, s, l), (_, gb, gl) in zip(preds_, ev):
            k = s >= thr
            t_, f_, n_ = label_errors_ref(b[k], s[k], l[k], gb, gl)
            tp, fp, fn_ = tp + t_, fp + f_, fn_ + n_
        rows.append((thr, tp / max(tp + fp, 1), tp / max(tp + fn_, 1)))
    curves[mode] = np.array(rows)
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for mode, col in [("text prompts", "#d93025"), ("visual prompts", "#1a73e8")]:
    c_ = curves[mode]
    axes[0].plot(c_[:, 2], c_[:, 1], "o-", color=col, label=mode)
    axes[1].plot(c_[:, 0], c_[:, 1], "-", color=col, label=f"precision, {mode}"); axes[1].plot(c_[:, 0], c_[:, 2], "--", color=col, label=f"recall, {mode}")
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision"); axes[0].set_xlim(0, 1.02); axes[0].set_ylim(0, 1.02); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title("precision-recall (class must be correct, IoU >= 0.5)")
axes[1].set_xlabel("confidence threshold"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_title("vs threshold")
fig.suptitle(f"Figure 6.3 - Text vs visual prompts on {N_EVAL} conveyor scenes", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. An auto-labelling pipeline with human review

**Auto-labelling** (pre-labelling, model-assisted labelling) = let a foundation model propose the labels, and spend human time only where the
model is unsure. The pipeline used by labelling tools (CVAT, Label Studio, Roboflow, Ultralytics' auto-annotate) is:

1. prompt an open-vocabulary detector (text or a few visual examples) → boxes with confidences (masks from SAM, Week 11B, if needed);
2. **accept** boxes with confidence $\ge t_\text{hi}$, **drop** boxes below $t_\text{lo}$;
3. send every image that has a box **between** $t_\text{lo}$ and $t_\text{hi}$ to a **human reviewer**, who corrects it;
4. export in YOLO format and train (Week 9B); use the trained model to pre-label the next batch (the data engine).

Here the "human" is the ground truth of `partsim`, so we can measure what each policy costs and how clean the labels become.

In [ ]:
if SMOKE:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 12, 6, 1, 320, 4
elif HW["cuda"]:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 200, 60, 30, 640, int(16 * HW["batch_scale"])
else:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 120, 40, 25, 416, 16
ROOT = data_dir("w13a")
tr_rng = np.random.default_rng(100)
train_scenes = [partsim.make_scene(tr_rng.integers(1 << 31), exposure=tr_rng.uniform(0.6, 1.1)) for _ in range(N_TRAIN)]
t0 = time.perf_counter()
auto = [predict_visual(img, conf=0.05)[:3] for img, _, _ in train_scenes]         # (boxes, scores, labels) per image
t_label = time.perf_counter() - t0
print(f"auto-labelled {N_TRAIN} images in {t_label:.0f} s ({t_label / N_TRAIN:.2f} s per image); "
      f"{sum(len(b) for b, _, _ in auto)} candidate boxes, {sum(len(b) for _, b, _ in train_scenes)} true objects")


def apply_policy(t_lo, t_hi):
    """Return the final labels per image, the reviewed image indices, and the label errors (TP, FP, FN) against the truth."""
    final, reviewed, tot = [], [], np.zeros(3, int)
    for i, ((b, s, l), (_, gb, gl)) in enumerate(zip(auto, train_scenes)):
        if np.any((s >= t_lo) & (s < t_hi)):
            reviewed.append(i); fb, fl = np.asarray(gb, float).reshape(-1, 4), np.asarray(gl, int)          # the human fixes it
        else:
            k = s >= t_hi; fb, fl = b[k], l[k]
        final.append((fb, fl))
        tot += label_errors_ref(fb, np.ones(len(fb)), fl, gb, gl)
    return final, reviewed, tot


def f1(t):
    return 2 * t[0] / max(2 * t[0] + t[1] + t[2], 1)


rows = []
for t_hi in [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.9]:
    _, rev, tot = apply_policy(0.05, t_hi)
    rows.append((t_hi, len(rev) / N_TRAIN, f1(tot), tot[1] + tot[2]))
rows = np.array(rows)
rand_rng = np.random.default_rng(0)
_, _, tot0 = apply_policy(0.25, 0.25)
rand_rows = []
for frac in rows[:, 1]:
    pick = set(rand_rng.choice(N_TRAIN, int(round(frac * N_TRAIN)), replace=False))
    t_ = np.zeros(3, int)
    for i, ((b, s, l), (_, gb, gl)) in enumerate(zip(auto, train_scenes)):
        k = s >= 0.25
        t_ += label_errors_ref(*((gb, np.ones(len(gb)), gl) if i in pick else (b[k], s[k], l[k])), gb, gl)
    rand_rows.append(f1(t_))
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].plot(rows[:, 1] * 100, rows[:, 2], "o-", color="#1a73e8", lw=2.5, label="review the uncertain images (confidence band)")
axes[0].plot(rows[:, 1] * 100, rand_rows, "s--", color="#9aa0a6", label="review the same number of RANDOM images")
for t_hi, frac, f, _ in rows[1::2]:
    axes[0].annotate(f"t_hi={t_hi:.2f}", (frac * 100, f), textcoords="offset points", xytext=(4, -12), fontsize=8)
axes[0].set_xlabel("images reviewed by a human (%)"); axes[0].set_ylabel("label F1 against the truth"); axes[0].grid(alpha=0.3); axes[0].legend(loc="lower right")
axes[0].set_title("label quality vs human effort")
r35 = rows[rows[:, 0] == 0.35][0]
axes[1].bar(["auto only\n(0% reviewed)", f"auto + review, t_hi=0.35\n({r35[1]:.0%} reviewed)", "human\n(100% labelled)"],
            [f1(tot0), r35[2], 1.0], color=["#f29900", "#1a73e8", "#34a853"])
axes[1].set_ylim(0.5, 1.02); axes[1].set_ylabel("label F1"); axes[1].grid(alpha=0.3, axis="y")
axes[1].set_title(f"auto only: {tot0[1]} wrong and {tot0[2]} missing boxes out of {tot0[0] + tot0[2]}")
fig.suptitle("Figure 7.1 - Spending human time where the model is unsure beats random checking", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎛️ Interactive: choose your review policy

In [ ]:
def policy_demo(t_lo=0.05, t_hi=0.35):
    t_hi = max(t_hi, t_lo)
    _, rev, tot = apply_policy(t_lo, t_hi)
    i = rev[0] if rev else 0
    img, gb, gl = train_scenes[i]
    b, s, l = auto[i]
    out = img.copy()
    for (x0, y0, x1, y1), sc, c in zip(b.astype(int), s, l):
        if sc < t_lo:
            continue
        col = (0, 200, 0) if sc >= t_hi else (0, 165, 255)
        cv2.rectangle(out, (x0, y0), (x1, y1), col, 2); cv2.putText(out, f"{NAMES[c]} {sc:.2f}", (x0, max(12, y0 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    fig, ax = plt.subplots(figsize=(9, 6.5))
    ax.imshow(rgb(out)); ax.axis("off")
    ax.set_title(f"review {len(rev)}/{N_TRAIN} images ({len(rev) / N_TRAIN:.0%}) -> label F1 {f1(tot):.3f}, {tot[1]} wrong, {tot[2]} missing\n"
                 f"image {i}: green = accepted, orange = send to reviewer" + ("" if rev else " (no image needs review)"))
    plt.show()


interact(policy_demo, t_lo=widgets.FloatSlider(value=0.05, min=0.01, max=0.5, step=0.01), t_hi=widgets.FloatSlider(value=0.35, min=0.05, max=0.95, step=0.05))

## 8. Does training on auto-labels work?

The real test of a label set is the model trained on it. We train two YOLO11n detectors (Week 9B) with identical settings — one on the human
(ground-truth) labels, one on the **auto-labels with no review at all** (confidence ≥ 0.25) — and evaluate both on the same human-labelled
validation set. Start this cell and continue reading Section 9 while it runs.

In [ ]:
def yolo_lines(boxes, labels, w, h):
    return [f"{int(c)} {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} {(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}" for b, c in zip(boxes, labels)]


def write_dataset(name, train_labels):
    root = ROOT / name
    shutil.rmtree(root, ignore_errors=True)
    for split, scenes, labels in [("train", train_scenes, train_labels), ("val", val_scenes, [(b, l) for _, b, l in val_scenes])]:
        (root / "images" / split).mkdir(parents=True); (root / "labels" / split).mkdir(parents=True)
        for i, ((img, _, _), (b, l)) in enumerate(zip(scenes, labels)):
            cv2.imwrite(str(root / "images" / split / f"{i:04d}.jpg"), img)
            (root / "labels" / split / f"{i:04d}.txt").write_text("\n".join(yolo_lines(b, l, img.shape[1], img.shape[0])) + "\n")
    (root / "data.yaml").write_text(f"path: {root.resolve()}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
    return root / "data.yaml"


va_rng = np.random.default_rng(200)
val_scenes = [partsim.make_scene(va_rng.integers(1 << 31)) for _ in range(N_VAL)]
label_sets = {"human labels": [(b, l) for _, b, l in train_scenes],
              "auto-labels (4 clicks, no review)": [(b[s >= 0.25], l[s >= 0.25]) for b, s, l in auto]}
scores = {}
for name, labels in label_sets.items():
    yaml_path = write_dataset(name.split()[0], labels)
    det = YOLO(str(download("https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt", WEIGHTS / "yolo11n.pt")))
    t0 = time.perf_counter()
    det.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=HW["num_workers"], flipud=0.5,
              project=str(ROOT / "runs"), name=name.split()[0], exist_ok=True, plots=False, verbose=False, seed=0, deterministic=True, amp=HW["amp"])
    m = det.val(data=str(yaml_path), imgsz=IMGSZ, device=DEVICE, plots=False, verbose=False)
    scores[name] = (m.box.map50, m.box.map, time.perf_counter() - t0)
    print(f"{name:<36} mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | {scores[name][2] / 60:.1f} min")
fig, ax = plt.subplots(figsize=(10, 4.5))
xs = np.arange(2)
for k, (name, col) in enumerate(zip(scores, ["#34a853", "#f29900"])):
    ax.bar(xs + (k - 0.5) * 0.38, scores[name][:2], 0.38, color=col, label=name)
    for x, v in zip(xs + (k - 0.5) * 0.38, scores[name][:2]):
        ax.text(x, v + 0.01, f"{v:.3f}", ha="center")
ax.set_xticks(xs); ax.set_xticklabels(["mAP50", "mAP50-95"]); ax.set_ylim(0, 1.25); ax.legend(loc="upper center", ncol=2); ax.grid(alpha=0.3, axis="y")
ax.set_title(f"Figure 8.1 - YOLO11n trained on {N_TRAIN} scenes, evaluated on the same human-labelled validation set", fontweight="bold")
plt.show()

**Interpretation.** With a clean, consistent domain, auto-labels from four example boxes get most of the way to human labels; the remaining gap
comes mostly from missed objects (false negatives teach the model that a part is background — Week 9A's label noise). Reviewing only the
**uncertain** images (Section 7) removes most label errors while a human still checks far fewer boxes than full manual labelling — and checking a
proposed box is several times faster than drawing one. On messy real data the gap is larger and review is essential.

## 9. Generative VLMs: captions, questions, structured answers

**Generative VLMs** (LLaVA, Qwen2.5-VL / Qwen3-VL, Gemma 3, SmolVLM, Florence-2, GPT-4o/5, Gemini, Claude) put a vision encoder (often a
CLIP/SigLIP ViT) in front of a **large language model**: image patches are projected into "visual tokens" that the LLM reads together with your
question, and it **generates** text — a caption, an answer, a JSON record, even box coordinates.

| capability | example use in a factory | risk |
|---|---|---|
| captioning | describe an anomaly for the shift report | generic or invented detail |
| visual question answering | "Is the guard door open?" | confident wrong answers |
| structured output (JSON) | fill an inspection record | format errors, made-up fields |
| grounding (boxes/points) | "point to the loose cable" | poor localisation precision |
| OCR / reading | read a label or a gauge | misread digits |

**Hallucination** — fluent text not supported by the image — is the central failure mode: the language model prefers plausible sentences.
Below we run **SmolVLM-256M** (Hugging Face, 256 M parameters — small enough for a laptop CPU) and check its answers against the ground truth
we know from `partsim`.

In [ ]:
def load_smolvlm(model_id="HuggingFaceTB/SmolVLM-256M-Instruct"):
    from transformers import AutoModelForImageTextToText, AutoProcessor
    proc = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.float16 if HW["cuda"] else torch.float32).to(HW["device"]).eval()
    return proc, model


def ask(img_bgr, question, max_new_tokens=60):
    from PIL import Image
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = vlm_proc.apply_chat_template(messages, add_generation_prompt=True)
    inputs = vlm_proc(text=prompt, images=[Image.fromarray(rgb(img_bgr).copy())], return_tensors="pt").to(HW["device"], dtype=vlm.dtype)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return vlm_proc.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()


try:
    t0 = time.perf_counter()
    vlm_proc, vlm = load_smolvlm()
    print(f"SmolVLM loaded in {time.perf_counter() - t0:.0f} s")
except Exception as exc:                                        # no internet / Hugging Face blocked
    if os.environ.get("CI"):
        raise
    vlm = None
    print(f"SmolVLM could not be loaded ({type(exc).__name__}: {exc}).\nRun this section in Google Colab, or on a network that can reach huggingface.co.")

In [ ]:
if vlm is not None:
    scene_q, sb_, sl_ = partsim.make_scene(21, n_objects=4)
    tests = [(GALLERY["bus"], "Describe this image in one sentence."),
             (GALLERY["bus"], "How many people are visible? Answer with a number."),
             (scene_q, "Describe this image in one sentence."),
             (scene_q, f"How many objects are on the belt? Answer with a number."),
             (scene_q, 'Return JSON: {"n_objects": int, "belt_colour": str, "defects_visible": bool}'),
             (GALLERY["part scratch"], "Is there a scratch on this part? Answer yes or no.")]
    answers = []
    t0 = time.perf_counter()
    for img, q in tests:
        answers.append(ask(img, q, 25 if SMOKE else 60))
    print(f"{len(tests)} answers in {time.perf_counter() - t0:.0f} s")
    fig, axes = plt.subplots(2, 3, figsize=(22, 10))
    for ax, (img, q), a in zip(axes.ravel(), tests, answers):
        ax.imshow(rgb(img)); ax.axis("off")
        ax.set_title("Q: " + q[:70] + ("..." if len(q) > 70 else "") + "\nA: " + a[:80].replace("\n", " "), fontsize=9, loc="left")
    fig.suptitle(f"Figure 9.1 - A 256 M-parameter VLM answering questions (truth for the conveyor: {len(sb_)} objects; the bus: 4 people)",
                 fontweight="bold")
    plt.tight_layout(); plt.show()

### Safeguards for VLMs in production
| risk | safeguard |
|---|---|
| hallucination | ask closed questions (yes/no, a number, a choice from a list); verify with a grounded model (detector, measurement) |
| format errors | constrained / JSON-schema decoding; validate and retry |
| latency, cost | small models on the edge (SmolVLM, Florence-2) or batch processing, not in a 10 ms loop |
| non-determinism | greedy decoding (`do_sample=False`), fixed model version |
| privacy | run locally; do not send production images to a cloud API without approval |
| accountability | log the image, prompt, model version and answer; humans decide on critical actions |

The pattern that works: **VLMs for flexible perception and triage, deterministic vision (Weeks 1–12) for measurement and control.**

### Optional: compare with OpenAI CLIP (Hugging Face)
`load_clip("openai")` loads OpenAI's CLIP ViT-B/32 with the same interface. Re-run Sections 2–4 with it (E4) and compare. It runs here only if
Hugging Face is reachable.

In [ ]:
try:
    openai_clip = load_clip("openai")
    p_ = (openai_clip.encode_images(def_te_img) @ openai_clip.encode_text(DEFECT_PROMPTS).T).argmax(1)
    c_ = (openai_clip.encode_images(list(crops)) @ openai_clip.encode_text([f"a photo of a {n}." for n in PART_NAMES["short names"]]).T).argmax(1)
    print(f"{openai_clip.name}: part crops {np.mean(c_ == crop_y):.0%} | defects {np.mean(p_ == def_te_y):.0%}  "
          f"(MobileCLIP: {results[('short names', 'a photo of a {}.')]:.0%} | {zs_acc:.0%})")
except Exception as exc:
    if os.environ.get("CI"):
        raise
    print(f"OpenAI CLIP not available here ({type(exc).__name__}).")

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Zero-shot probabilities
Return the $N \times C$ matrix of class probabilities $\mathrm{softmax}(\lambda\, U V^\top)$ for unit image embeddings `img_emb` ($N\times D$, or a single
$D$ vector) and text embeddings `txt_emb` ($C \times D$). Subtract the row maximum before `np.exp` so that large scales do not overflow.

In [ ]:
def zero_shot_probs(img_emb, txt_emb, scale):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    u = np.array([[1.0, 0.0], [0.0, 1.0]]); v = np.array([[1.0, 0.0], [0.6, 0.8]])
    p = zero_shot_probs(u, v, 10.0)
    e = np.exp(10 * (u @ v.T)); ref = e / e.sum(1, keepdims=True)
    assert p.shape == (2, 2) and np.allclose(p, ref), "wrong probabilities"
    assert np.all(np.isfinite(zero_shot_probs(u, v, 5000.0))), "overflow: subtract the row maximum first"
    assert zero_shot_probs(u[0], v, 10.0).shape == (1, 2), "a single embedding should give a (1, C) result"
    assert np.allclose(zero_shot_probs(G_EMB, T_EMB, clip_model.logit_scale), zero_shot_probs_ref(G_EMB, T_EMB, clip_model.logit_scale), atol=1e-6)


check("E1 zero_shot_probs", _test_e1)

### E2. Prompt ensembling
For each class name, encode every template (`t.format(name)`) with `encode_text`, normalise each embedding, average them, and renormalise. Return a $C \times D$ array.

In [ ]:
def prompt_ensemble(encode_text, classnames, templates):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    table = {"a x": [3.0, 0.0], "the x": [0.0, 4.0], "a y": [0.0, -2.0], "the y": [0.0, -5.0]}
    fake = lambda texts: np.array([table[t] for t in texts])
    e = prompt_ensemble(fake, ["x", "y"], ["a {}", "the {}"])
    assert e.shape == (2, 2), f"expected shape (2, 2), got {np.shape(e)}"
    assert np.allclose(e[0], [np.sqrt(0.5), np.sqrt(0.5)]), "normalise each template embedding BEFORE averaging"
    assert np.allclose(e[1], [0.0, -1.0]), "renormalise the mean"
    assert np.allclose(prompt_ensemble(clip_model.encode_text, ["washer"], TEMPLATES[:3]),
                       prompt_ensemble_ref(clip_model.encode_text, ["washer"], TEMPLATES[:3]), atol=1e-5)


check("E2 prompt_ensemble", _test_e2)

### E3. Checking auto-labels
Return `(tp, fp, fn)` for one image. Go through the predictions **in order of decreasing score**; a prediction is a true positive if some
**not yet matched** ground-truth box of the **same class** has IoU ≥ `iou_thr` with it (match it to the one with the highest IoU). Use `box_iou`
from Section 6.

In [ ]:
def label_errors(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_thr=0.5):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    gt = np.array([[0, 0, 10, 10], [20, 0, 30, 10]]); gl = np.array([0, 1])
    pb = np.array([[0, 0, 10, 10], [1, 0, 11, 10], [20, 0, 30, 10], [50, 50, 60, 60]]); pl = np.array([0, 0, 0, 1]); ps = np.array([0.9, 0.8, 0.7, 0.6])
    assert tuple(label_errors(pb, ps, pl, gt, gl)) == (1, 3, 1), "duplicates, wrong classes and background boxes are false positives"
    assert tuple(label_errors(pb[:0], ps[:0], pl[:0], gt, gl)) == (0, 0, 2), "no predictions: every object is missed"
    g2, a_b = np.array([[0, 0, 10, 10], [4, 0, 14, 10]]), np.array([[4, 0, 14, 10], [3, 0, 13, 10]])
    assert tuple(label_errors(a_b, np.array([0.3, 0.9]), np.array([0, 0]), g2, np.array([0, 0]))) == (1, 1, 1),         "process predictions in order of decreasing score, each taking its best unmatched box"
    for (b, s, l), (_, g, c) in list(zip(auto, train_scenes))[:5]:
        assert tuple(label_errors(b, s, l, g, c)) == tuple(label_errors_ref(b, s, l, g, c)), "differs from Section 6 on the auto-labels"


check("E3 label_errors", _test_e3)

### E4. Beat the defect baseline (no self-check)
Zero-shot defect accuracy was about a third (Figure 3.2). Try (a) better prompts and prompt ensembles, (b) OpenAI CLIP (`load_clip("openai")`),
(c) **cropping** the part rim before encoding (the defects are small), (d) the linear probe with k = 5. Report a table: method, accuracy, labels
needed. Which is worth it in a factory, and why?

In [ ]:
# E4: your code here

## 11. Challenge: auto-label your own objects
Take 30 photos of 2–3 kinds of small objects on your desk (pens, coins, keys). Label **one box per class** in one photo (any tool, or by reading
coordinates in matplotlib), auto-label the rest with YOLOE visual prompts, add masks with SAM (Week 11B), review the uncertain ones, export in YOLO
format and train YOLO11n (Week 9B). Report: how long did labelling take compared with labelling every box yourself, and what mAP did you reach?

In [ ]:
# Your challenge code here

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": "What does CLIP's contrastive loss push together?",
        "options": [
            'All images',
            'Each image and its own caption, apart from the other captions in the batch',
            'Images of the same size',
            'Words of the same length',
        ],
        "answer": '8aaf8cd8be',
        "explain": 'InfoNCE: the diagonal of the similarity matrix should win.',
    },
    {
        "q": 'In zero-shot classification, what plays the role of the classifier weights?',
        "options": [
            'Random vectors',
            'The text embeddings of the class prompts',
            'The image pixels',
            'The logit scale',
        ],
        "answer": '9c061602fc',
        "explain": 'Classes are written in language, not learned from labels.',
    },
    {
        "q": 'What is prompt ensembling?',
        "options": [
            'Training several models',
            'Averaging the normalised text embeddings of several templates per class',
            'Using several images',
            'Asking a chatbot',
        ],
        "answer": '66c9411638',
        "explain": 'It reduces sensitivity to one wording.',
    },
    {
        "q": 'Why is zero-shot CLIP weak on subtle industrial defects?',
        "options": [
            'Images are too large',
            'Such fine, domain-specific states are rare in web captions',
            'CLIP cannot see grey images',
            'The logit scale is too small',
        ],
        "answer": '3571d9db1f',
        "explain": 'Close the gap with a few labels (linear probe) or a dedicated method.',
    },
    {
        "q": 'What is a visual prompt in YOLOE?',
        "options": [
            'A text description',
            'An example box in a reference image that defines a class',
            'A colour filter',
            'A mask for SAM',
        ],
        "answer": 'ee5bb9baac',
        "explain": 'Useful when words do not describe your objects well.',
    },
    {
        "q": 'In a review policy, which images should a human check first?',
        "options": [
            'Random ones',
            'Those with detections of intermediate confidence',
            'Only the most confident ones',
            'None',
        ],
        "answer": '24f6346262',
        "explain": 'Confident boxes are accepted, very low ones dropped; the uncertain band is where errors are.',
    },
    {
        "q": 'What is a VLM hallucination?',
        "options": [
            'A slow answer',
            'Fluent output not supported by the image',
            'A JSON error',
            'A blurry image',
        ],
        "answer": 'e76db531cd',
        "explain": 'Mitigate with closed questions and grounded verification.',
    },
    {
        "q": 'Which task should NOT rely on a generative VLM alone?',
        "options": [
            'Drafting a shift report',
            'Searching logged images',
            'Measuring a part to 0.1 mm for accept/reject',
            'Triage of unusual images',
        ],
        "answer": '73d8690ba2',
        "explain": 'Use calibrated, deterministic measurement (Weeks 2-4, 12A).',
    },
], title='Week 13A quiz')

## 13. Exit ticket (reflection)
1. Give one inspection task where zero-shot CLIP would be enough and one where it would fail. Why?
2. Your auto-labeller reached label F1 0.93. What would you do before training on those labels?
3. A colleague proposes a chatbot VLM to decide pass/fail on the line. What do you answer?

## 14. What's new (2025–26)

> - **Encoders**: SigLIP 2 (Google, 2025), Meta's Perception Encoder and DINOv3 (2025), MobileCLIP 2 — better dense features for detection and segmentation.
> - **Open-vocabulary detection and segmentation**: YOLOE and YOLO-World in the YOLO ecosystem; Grounding DINO 1.5/1.6, OWLv2; **SAM 3** (Meta, late 2025)
>   segments *all* instances of a text or image concept — auto-labelling for masks in one step.
> - **Generative VLMs** small enough for edge devices: SmolVLM2, Qwen2.5-VL / Qwen3-VL (3–8 B), Gemma 3, Florence-2; many support boxes/points and JSON output.
> - **Labelling tools** integrate all of the above (CVAT, Label Studio, Roboflow Auto Label, Ultralytics `auto_annotate`), making the review loop of Section 7 the norm.

## 15. References and further exploration
- Radford, A. et al. (2021). *Learning transferable visual models from natural language supervision* (CLIP). ICML. [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)
- Oord, A. van den, Li, Y. & Vinyals, O. (2018). *Representation learning with contrastive predictive coding* (InfoNCE). [arXiv:1807.03748](https://arxiv.org/abs/1807.03748)
- Vasu, P. K. A. et al. (2024). *MobileCLIP: Fast image-text models through multi-modal reinforced training.* CVPR. [arXiv:2311.17049](https://arxiv.org/abs/2311.17049)
- Zhai, X. et al. (2023). *Sigmoid loss for language image pre-training* (SigLIP). ICCV. [arXiv:2303.15343](https://arxiv.org/abs/2303.15343)
- Liang, W. et al. (2022). *Mind the gap: understanding the modality gap in multi-modal contrastive representation learning.* NeurIPS. [arXiv:2203.02053](https://arxiv.org/abs/2203.02053)
- Wang, A. et al. (2025). *YOLOE: Real-time seeing anything.* ICCV. [arXiv:2503.07465](https://arxiv.org/abs/2503.07465)
- Cheng, T. et al. (2024). *YOLO-World: Real-time open-vocabulary object detection.* CVPR. [arXiv:2401.17270](https://arxiv.org/abs/2401.17270)
- Liu, S. et al. (2024). *Grounding DINO: Marrying DINO with grounded pre-training for open-set object detection.* ECCV. [arXiv:2303.05499](https://arxiv.org/abs/2303.05499)
- Minderer, M. et al. (2023). *Scaling open-vocabulary object detection* (OWLv2). NeurIPS. [arXiv:2306.09683](https://arxiv.org/abs/2306.09683)
- Liu, H. et al. (2023). *Visual instruction tuning* (LLaVA). NeurIPS. [arXiv:2304.08485](https://arxiv.org/abs/2304.08485)
- Marafioti, A. et al. (2025). *SmolVLM: Redefining small and efficient multimodal models.* [arXiv:2504.05299](https://arxiv.org/abs/2504.05299)
- Li, Y. et al. (2023). *Evaluating object hallucination in large vision-language models* (POPE). EMNLP. [arXiv:2305.10355](https://arxiv.org/abs/2305.10355)
- Ultralytics docs: [YOLOE](https://docs.ultralytics.com/models/yoloe/) · [auto-annotation](https://docs.ultralytics.com/reference/data/annotator/)

**Data:** scikit-image sample images; OpenCV sample data; `bus.jpg` and `zidane.jpg` shipped with Ultralytics; conveyor scenes and parts from `partsim.py`.

## 16. Key takeaways
- CLIP-style models embed images and text in one space; **cosine similarity** drives zero-shot classification, retrieval and open-vocabulary detection.
- **Words matter**: templates and ensembles change accuracy by tens of percent; validate prompts like any other hyper-parameter.
- Zero-shot is strong on everyday **categories** and weak on subtle **industrial defects**; a **few labels + a linear probe** close much of the gap.
- **Auto-labelling** with open-vocabulary detectors plus **confidence-based human review** gives near-human labels at a fraction of the cost.
- Generative VLMs are flexible but **hallucinate**; use closed questions, grounded checks and humans for critical decisions.

**Next session (13B):** generative models and **synthetic data** — creating training images instead of collecting them.